# Dreamer en el ambiente de la galleta

Entrena [r2dreamer-cookie](https://github.com/FabriRandon/r2dreamer-cookie) en Colab, guardando en Drive.

Si la sesión se corta, vuelve a correr las celdas 1 a 5 y después la 7 con el mismo `LOGDIR`: el run se retoma solo desde el último checkpoint.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.

## 1. Revisar la GPU que tocó

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Montar Drive

Aquí van los checkpoints, para que sobrevivan al corte de sesión.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clonar el repo (o actualizarlo si ya está)

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 4. Instalar

r2dreamer necesita Python 3.11 y Colab suele traer una versión más nueva, así que se crea un entorno aparte con `uv`. Toma unos minutos la primera vez; hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
cd /content/r2dreamer-cookie
$HOME/.local/bin/uv venv --python 3.11 .venv
$HOME/.local/bin/uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

## 5. Comprobar que todo quedó bien

In [ ]:
!cd /content/r2dreamer-cookie && .venv/bin/python -c "\
import torch, gymnasium, cookie_env; \
from envs.cookie import Cookie; \
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU'); \
print('float16 ok:', bool(torch.isfinite((torch.randn(256, 256, device=\"cuda\", dtype=torch.float16) @ torch.randn(256, 256, device=\"cuda\", dtype=torch.float16)).sum()))); \
print('galleta ok:', Cookie('partial').reset()['image'].shape)"

## 6. Medir la velocidad (opcional, unos 10 minutos)

Corre un rato en disco local y estima cuánto tardaría un run completo de 1M de pasos. Vale la pena la primera vez, para saber con qué GPU te tocó trabajar.

In [ ]:
!cd /content/r2dreamer-cookie && timeout 600 .venv/bin/python -u train.py \
    env=cookie env.task=cookie_partial env.eval_episode_num=0 \
    model.compile=False trainer.update_log_every=500 \
    logdir=/content/prueba_velocidad > /content/prueba_velocidad.log 2>&1

In [ ]:
import json, pathlib, subprocess, csv, datetime

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
pasos = [json.loads(l) for l in open("/content/prueba_velocidad/metrics.jsonl") if '"fps/fps"' in l]
fps = [p["fps/fps"] for p in pasos if p["fps/fps"] > 0]

if not fps:
    print("No alcanzó a registrar velocidad; dale más tiempo a la celda anterior.")
else:
    v = sum(fps) / len(fps)
    horas = 1e6 / v / 3600
    print(f"{gpu}: {v:.1f} pasos por segundo → 1M de pasos en unas {horas:.1f} horas")

    # Se va acumulando una tabla en Drive, para comparar las GPU que vayan tocando.
    tabla = pathlib.Path("/content/drive/MyDrive/galleta/velocidades.csv")
    tabla.parent.mkdir(parents=True, exist_ok=True)
    nueva = not tabla.exists()
    with open(tabla, "a", newline="") as f:
        w = csv.writer(f)
        if nueva:
            w.writerow(["fecha", "gpu", "pasos_por_segundo", "horas_por_millon"])
        w.writerow([datetime.date.today().isoformat(), gpu, round(v, 2), round(horas, 1)])
    print("\n", tabla.read_text())

## 7. El entrenamiento de verdad

`LOGDIR` va en Drive. Para retomar un run cortado, corre esta misma celda sin cambiar nada.

El buffer se guarda junto con el modelo, porque Dreamer lo necesita para seguir aprendiendo. `buffer.max_size=1e5` son unos 2 GB por checkpoint: no es por espacio en Drive, es porque escribir ahí es lento. Si ves que guarda rápido, súbelo.

Variantes disponibles en `env.task`: `cookie_partial` (la galleta con vista parcial, el caso que falla), `cookie_deterministic`, `cookie_norespawn`, `cookie_full` y `cookie_fullfixed`.

In [ ]:
LOGDIR = "/content/drive/MyDrive/galleta/cookie_partial_01"

!cd /content/r2dreamer-cookie && .venv/bin/python -u train.py \
    env=cookie env.task=cookie_partial \
    buffer.max_size=1e5 trainer.save_every=2e4 \
    logdir="{LOGDIR}"

## 8. Ver las curvas

Se puede abrir mientras entrena, en otra pestaña del mismo notebook, o después apuntando a un run viejo de Drive.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/galleta"